In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
# from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# load MNIST data
transform = transforms.ToTensor()
train_data  = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)


In [ ]:
# firstly see how the data look like and size of dataset
img, label = train_data[16]
print(img.size)
# plt.imshow(img)
print(f"label is {label}, and image size is {img.size}\n")
print(f"train data size {len(train_data)}, test data size {len(test_data)}")

<built-in method size of Tensor object at 0x7b5548874870>
label is 2, and image size is <built-in method size of Tensor object at 0x7b5548874870>

train data size 60000, test data size 10000


In [ ]:
# process data get a train with 20 points, valid with 100 points rest of train are pooling points
# train and valid need random and balanced
len_train = len(train_data)
index_lst = []
for i in range(10):
  data_index = [index for index in range(len_train) if train_data[index][1] == i ]
  index_lst.append(data_index)

In [ ]:
import random
train_index = []
valid_index = []
for i in range(10):
  temp_choose = random.sample(index_lst[i], k=12)
  train_index.extend(temp_choose[:2])
  valid_index.extend(temp_choose[2:])
  index_lst[i] = [x for x in index_lst[i] if x not in temp_choose]


In [ ]:
pooling_index = [i for j in  index_lst for i in j]
len(pooling_index)

59880

In [ ]:
class NewDataset(Dataset):
  def __init__(self, index):
    imgs = []
    labels = []
    for i in index:
      img, label = train_data[i]
      imgs.append(img)
      labels.append(label)
    self.imgs = torch.stack(imgs)
    self.labels = torch.tensor(labels)

  def __len__(self):
    return len(self.imgs)

  def __getitem__(self, idx):
    return self.imgs[idx], self.labels[idx]



In [ ]:
class BaseCNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.covn1 =nn.Conv2d(in_channels=1, out_channels=32,  kernel_size=4)
    self.covn2 = nn.Conv2d(in_channels=32, out_channels=32,  kernel_size=4)
    self.max_pool = nn.MaxPool2d(kernel_size=2)
    self.dropout_layer1 = nn.Dropout(p=0.25)
    self.flatten = nn.Flatten()
    self.dense_layer1 = nn.Linear(in_features=3872, out_features=128)
    self.dropout_layer2 = nn.Dropout(p=0.5)
    self.dense_layer2 = nn.Linear(in_features=128, out_features=10)

  def forward(self, x):
    x = F.relu(self.covn1(x))
    x = F.relu(self.covn2(x))
    x = self.max_pool(x)
    x = self.dropout_layer1(x)
    x = self.flatten(x)
    x = F.relu(self.dense_layer1(x))
    x = self.dropout_layer2(x)
    x = self.dense_layer2(x)
    # x = F.softmax(x, dim=1)
    return x

In [ ]:
# write train function
def train_model(trainData):
    train_loader  = DataLoader(trainData, batch_size=120, shuffle=True)
    model = BaseCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    weight_decay = 0.02/(len(trainData))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    # Training loop
    for epoch in tqdm.tqdm(range(50)):
      total_loss = 0
      for i, (imgs, labels) in (enumerate(train_loader)):
        optimizer.zero_grad()
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
      avg_loss = total_loss / len(train_loader)
      # if epoch % 79 == 0:
      #   print(f"/n epoch{epoch}, train_loss is {avg_loss:.4f}.")

    return model



In [ ]:
def max_entropy(model, poolingData, pooling_index):
  # calculate top k
  # print(len(poolingData))
  batch_size = 200
  pooling_loader  = DataLoader(poolingData, batch_size=200, shuffle=False)
  total_entropy = torch.tensor([]).to(device)
  drop_out_iter = 100
  model.train()
  for i ,(imgs, labels) in (enumerate(pooling_loader)):
    num_data = len(labels)
    # print(num_data)
    pred_prob = torch.zeros(num_data, 10).to(device)
    for i in range(drop_out_iter):
      with torch.no_grad():
        output = model(imgs.to(device))
      pred_prob += F.softmax(output, dim=1)
    prob_out = pred_prob / drop_out_iter
    log_out = torch.log2(prob_out)
    max_en = prob_out * log_out
    max_en = -torch.sum(max_en, dim=1)
    if i == 0 :
      total_entropy = max_en
    else:
      total_entropy = torch.cat((total_entropy, max_en), dim=0)
  # print(total_entropy.shape)
  top_k_value, top_k_idx = torch.topk(total_entropy, k=10, dim=0)

  new_data_index = []
  for i in top_k_idx:
    new_data_index.append(pooling_index[i])
  # print(f"set is {set(top_k_idx.tolist())}")
  new_pooling_index  = [v for i, v in enumerate(pooling_index) if i not in set(top_k_idx.tolist()) ]
  return  new_data_index, new_pooling_index

In [ ]:
# calculate test accuracy
def test_accuracy( model, test_data, device=device):
  test_loader = DataLoader(test_data, batch_size=200, shuffle=False)
  model.eval()
  correct = 0
  n_data = len(test_data)
  with torch.no_grad():
    for imgs, labels in test_loader:
      imgs = imgs.to(device)
      labels = labels.to(device)
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
  return correct / n_data

In [ ]:
# one experiment
#return pooling_index new train_index
def run_once(train_index, pooling_index, test_data=test_data):
  print(f"curr size of train_data {len(train_index)}, curr size of pooling data {len(pooling_index)}  ")
  trainData = NewDataset(index=train_index)
  # vaildData = NewDataset(index=valid_index)
  poolingData =  NewDataset(index=pooling_index)
  model = train_model(trainData=trainData)
  new_trainData_index, new_pool_index = max_entropy(model, poolingData, pooling_index)
  train_index.extend(new_trainData_index)
  test_ac = test_accuracy(model, test_data)
  print(f"test accuracy is {test_ac}")
  return train_index, new_pool_index, test_ac


In [ ]:
# number of experiement
pooling_index_temp = pooling_index.copy()
train_index_temp = train_index.copy()

n_experiement = 100
test_accuracy_lst = []
for i in range(n_experiement):
    train_index_temp, pooling_index_temp, test_ac= run_once(train_index_temp, pooling_index_temp)
    test_accuracy_lst.append(test_ac)



curr size of train_data 20, curr size of pooling data 59880  


100%|██████████| 50/50 [00:00<00:00, 405.99it/s]


test accuracy is 0.6183
curr size of train_data 30, curr size of pooling data 59870  


100%|██████████| 50/50 [00:00<00:00, 395.34it/s]


test accuracy is 0.5824
curr size of train_data 40, curr size of pooling data 59860  


...

test accuracy is 0.9783
curr size of train_data 1000, curr size of pooling data 58900  


100%|██████████| 50/50 [00:01<00:00, 38.88it/s]


test accuracy is 0.9809
curr size of train_data 1010, curr size of pooling data 58890  


100%|██████████| 50/50 [00:01<00:00, 38.09it/s]


test accuracy is 0.9779


In [ ]:
def save_accuracy(file_name, accuracy_lst):
  path = '/content/drive/MyDrive/UDL_DATA/'
  path = path + file_name
  with open(path, 'w') as f:
      for item in accuracy_lst:
          f.write(f"{item}\n")



In [ ]:
save_accuracy("maxEntropy_ex3.txt",test_accuracy_lst )